# Feature extraction and indexing for hybrid retrieval
Chunk embeddings are obtained using the [Multilingual E5 Base](https://huggingface.co/intfloat/multilingual-e5-base) model for semantic (dense) retrieval.

Terms are extracted from chunks for keyword-based (sparse) retrieval.

Chunks and their embeddings are stored in a Chroma database that builds an HNSW index.

A BM25 index is built from terms and saved to a file.

## 1. Feature extraction

### 1.1 Install libraries

In [ ]:
%pip install -U transformers
%pip install -U rank_bm25

### 1.2  Import libraries

In [ ]:
import json
import math
import re
import uuid
from datetime import datetime

import torch
from google.colab import drive
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModel

### 1.3 Functions for semantic feature extraction

These functions are used to obtain chunk embeddings.

In [ ]:
def mean_pool(last_hidden_states, attention_mask):
    """
    Computes a sequence embedding by mean-pooling the embeddings of all
    non-padding tokens.

    Args:
        last_hidden_states: Tensor of shape (batch_size, seq_len, hidden_dim).
        attention_mask: Tensor of shape (batch_size, seq_len).

    Returns:
        A tensor of shape (batch_size, hidden_dim).
    """

    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)

    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]


def get_embedding(text, tokenizer, model):
    """
    Generates a text embedding using a transformer encoder and mean pooling.

    Args:
        text: Input text.
        tokenizer: Tokenizer associated with the model.
        model: Transformer encoder model.

    Returns:
        A tensor containing the text embedding.
    """

    inputs = tokenizer(text, max_length=512, padding=True, truncation=True, return_tensors='pt')

    with torch.no_grad():
        outputs = model(**inputs)
    embedding = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])

    return embedding.squeeze()

### 1.4 Class for lexical feature extraction and indexing (BM25)
This class provides lexical indexing over text chunks for BM25-based retrieval.

It computes the term statistics used to score chunks for a given query.

In [ ]:
class BM25Log1(BM25Okapi):
    """
    BM25 variant with a non-negative IDF formulation.

    During initialization, this class builds an index containing the corpus
    statistics (term frequencies, chunk frequencies, chunk lengths, etc.)
    required to score chunks for a given query.

    Unlike the standard BM25 implementation, the IDF is computed as:

    ```
    IDF(t) = log(1 + (N - df(t) + 0.5) / (df(t) + 0.5))
    ```

    This formulation guarantees non-negative IDF values without penalizing
    very common terms.
    """

    def __init__(self, chunk_terms, chunk_ids, **kwargs):
        super().__init__(chunk_terms, **kwargs)

        self.chunk_ids = chunk_ids

        df = {}
        for doc in self.doc_freqs:
            for term in set(doc):
                df[term] = df.get(term, 0) + 1

        N = self.corpus_size
        for term, dfi in df.items():
            self.idf[term] = math.log(1 + (N - dfi + 0.5) / (dfi + 0.5))

    @staticmethod
    def extract_terms(text):
        """
        Extracts normalized terms from text for BM25 indexing and retrieval.

        Converts text to lowercase and extracts word-level terms
        (including basic accented Latin characters) using a regular expression.

        Args:
            text: Input text.

        Returns:
            List of normalized terms.
        """

        return re.findall(r"\b[a-zA-ZáéíóúñÁÉÍÓÚÑ]+\b", text.lower())


    def save(self, path):
        """
        Serializes the BM25 model state to a JSON file for later reconstruction.

        Args:
            path: Output file path.
        """
        data = {
            "chunk_ids": self.chunk_ids,
            "idf": {k: float(v) for k, v in self.idf.items()},
            "doc_freqs": self.doc_freqs,
            "doc_len": self.doc_len,
            "avgdl": self.avgdl,
            "corpus_size": self.corpus_size,
            "k1": self.k1,
            "b": self.b
        }

        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    @classmethod
    def load(cls, path):
        """
        Loads a serialized BM25 model state from a JSON file.

        Reconstructs the BM25 instance without recomputing corpus statistics,
        restoring document frequencies, document lengths, IDF values, and
        model hyperparameters.

        Args:
            path: Path to the JSON file.

        Returns:
            Reconstructed BM25 instance.
        """

        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        bm25 = cls.__new__(cls)
        bm25.chunk_ids = data["chunk_ids"]
        bm25.idf = {k: float(v) for k, v in data["idf"].items()}
        bm25.doc_freqs = data["doc_freqs"]
        bm25.doc_len = data["doc_len"]
        bm25.avgdl = data["avgdl"]
        bm25.corpus_size = data["corpus_size"]
        bm25.k1 = data["k1"]
        bm25.b = data["b"]

        return bm25

### 1.5 Hybrid (dense + sparse) feature extraction

In [ ]:
# Mount Google Drive for file access
drive.mount("/content/drive", force_remount=False)

# Model used for embedding generation: Multilingual E5 Base
model_name = "intfloat/multilingual-e5-base"

# Load the model and tokenizer from Hugging Face Hub
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Feature extraction metadata
feature_extraction_process = {
        "process_id": str(uuid.uuid4()),
        "timestamp": str(int(datetime.now().timestamp())),
        "model": model_name
    }

# Load the chunks file into memory
CHUNK_DIR = "/content/drive/MyDrive/RAG_UPC_Final_project"
chunks_file_name = "chunks_table_1781889339.json" # @param {type:"string"}
chunks_file_path = f"{CHUNK_DIR}/{chunks_file_name}"
with open(chunks_file_path, encoding="utf-8") as f:
    input = json.load(f)

# Enrich chunk info with embeddings and terms
chunking_process = input["chunking_process"]
chunk_info_list_in = input["chunks"]
chunk_info_list_out = []
chunk_terms = []
chunk_ids = []
for chunk_info in chunk_info_list_in:
    embedding = get_embedding(chunk_info["chunk_text"], tokenizer, model)
    terms = BM25Log1.extract_terms(chunk_info["chunk_text"])
    chunk_info.update({
        "embedding": embedding.detach().cpu().tolist(),
        "terms": terms})
    chunk_info_list_out.append(chunk_info)
    chunk_terms.append(terms)
    chunk_ids.append(chunk_info["chunk_id"])

# Output to be serialized
output = {
    "chunking_process": chunking_process,
    "feature_extraction_process": feature_extraction_process,
    "chunks": chunk_info_list_out
    }

# Save chunks to a JSON file
with open(chunks_file_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=4, ensure_ascii=False)

## 2. Indexing
Chunks are stored in a Chroma database, which builds an HNSW index for fast semantic (_dense_) retrieval.

A BM25 index is built over the chunks for fast keyword-based (_sparse_) retrieval, using a non-negative IDF variant to avoid negative contributions from very frequent terms.

The BM25 index is saved to a file.

### 2.1 Install libraries

In [ ]:
%pip install -U chromadb

### 2.2 Import libraries

In [ ]:
import chromadb

### 2.3 Indexing
Store chunks, embeddings, and metadata in ChromaDB; embeddings are indexed using HNSW.

Builds a BM25 index and saves it to a file.

In [ ]:

# Database file directory
DB_DIR = "/content/drive/MyDrive/RAG_UPC_Final_project"

# Database file path
tst = str(int(datetime.now().timestamp()))
db_file_name = f"chroma_db_{tst}"
db_file_path = f"{DB_DIR}/{db_file_name}"

# Create a persistent ChromaDB database
client = chromadb.PersistentClient(path=db_file_path)

# Delete collection if it exists (to avoid duplicate inserts on re-runs)
collection_name = "construction_site_visit_reports"
try:
    client.delete_collection(collection_name)
except Exception:
    pass

# Create a collection with cosine distance (HNSW index internally)
collection = client.create_collection(
    name=collection_name,
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
)

# Add chunks, their embeddings, and metadata to the vector database
collection.add(
    ids=[str(chunk_info["chunk_id"]) for chunk_info in chunk_info_list_out],
    documents=[chunk_info["chunk_text"] for chunk_info in chunk_info_list_out],
    embeddings=[chunk_info["embedding"] for chunk_info in chunk_info_list_out],
    metadatas=[{"docx_id": chunk_info["docx_id"],
                "chunk_id": chunk_info["chunk_id"],
                "chunk_index": chunk_info["chunk_index"]}
                for chunk_info in chunk_info_list_out]
)

# Build BM25 index from the corpus of chunk terms
bm25 = BM25Log1(chunk_terms, chunk_ids)

# BM25 index file directory
BM25_DIR = "/content/drive/MyDrive/RAG_UPC_Final_project"

# BM25 index file path (use the same tst as in the database filename)
bm25_file_name = f"bm25_{tst}.json"
bm25_file_path = f"{BM25_DIR}/{bm25_file_name}"

# Persist the BM25 index into a file
bm25.save(bm25_file_path)

# End
print(f"{collection_name = }")
print(f"Stored {collection.count()} records in ChromaDB.")
print(f"Stored {len(chunk_terms)} records in bm25.")
